<a href="https://colab.research.google.com/github/Juadrg/Estructuta-de-base-de-datos/blob/main/Tarea_de_arboles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
archivo co creado con inteligencia artificial

In [4]:
import random
import time

# El GRADO define la capacidad de los nodos en el Árbol B+
GRADO = 16

# ---------- PARTE 1: GENERAR DATOS ----------

def generar_estudiantes(n):
    estudiantes = []
    for i in range(n):
        estudiantes.append({
            "id": 1000 + i,
            "nombre": "Estudiante" + str(i),
            "promedio": round(random.uniform(1, 10), 1)
        })
    return estudiantes


# ---------- PARTE 2: ESTRUCTURA DEL ARBOL B+ ----------

class NodoBPlus:
    def __init__(self, hoja=False):
        self.hoja = hoja
        self.claves = []
        self.hijos = []
        self.siguiente = None

class BPlusTree:
    def __init__(self, grado=GRADO):
        self.raiz = NodoBPlus(True)
        self.grado = grado

    def insertar(self, est):
        raiz = self.raiz
        if len(raiz.claves) == self.grado - 1:
            nueva_raiz = NodoBPlus(False)
            self.raiz = nueva_raiz
            nueva_raiz.hijos.append(raiz)
            self._dividir_hijo(nueva_raiz, 0)
            self._insertar_no_lleno(nueva_raiz, est)
        else:
            self._insertar_no_lleno(raiz, est)

    def _insertar_no_lleno(self, nodo, est):
        i = len(nodo.claves) - 1
        if nodo.hoja:
            nodo.claves.append(None)
            while i >= 0 and est["id"] < nodo.claves[i]["id"]:
                nodo.claves[i+1] = nodo.claves[i]
                i -= 1
            nodo.claves[i+1] = est
        else:
            while i >= 0 and est["id"] < nodo.claves[i]:
                i -= 1
            i += 1
            if len(nodo.hijos[i].claves) == self.grado - 1:
                self._dividir_hijo(nodo, i)
                if est["id"] > nodo.claves[i]:
                    i += 1
            self._insertar_no_lleno(nodo.hijos[i], est)

    def _dividir_hijo(self, padre, i):
        nodo_lleno = padre.hijos[i]
        nuevo_nodo = NodoBPlus(nodo_lleno.hoja)
        mid = len(nodo_lleno.claves) // 2

        if nodo_lleno.hoja:
            clave_subir = nodo_lleno.claves[mid]["id"]
            nuevo_nodo.claves = nodo_lleno.claves[mid:]
            nodo_lleno.claves = nodo_lleno.claves[:mid]
            nuevo_nodo.siguiente = nodo_lleno.siguiente
            nodo_lleno.siguiente = nuevo_nodo
        else:
            clave_subir = nodo_lleno.claves[mid]
            nuevo_nodo.claves = nodo_lleno.claves[mid+1:]
            nuevo_nodo.hijos = nodo_lleno.hijos[mid+1:]
            nodo_lleno.claves = nodo_lleno.claves[:mid]
            nodo_lleno.hijos = nodo_lleno.hijos[:mid+1]

        padre.claves.insert(i, clave_subir)
        padre.hijos.insert(i + 1, nuevo_nodo)

    def buscar(self, id_buscar):
        nodo = self.raiz
        while not nodo.hoja:
            i = 0
            while i < len(nodo.claves) and id_buscar >= nodo.claves[i]:
                i += 1
            nodo = nodo.hijos[i]
        for e in nodo.claves:
            if e["id"] == id_buscar:
                return e
        return None


# ---------- PARTE 3: ESTRUCTURA DEL ARBOL ABB ----------

class NodoABB:
    def __init__(self, est):
        self.est = est
        self.izq = None
        self.der = None

class ABB:
    def __init__(self):
        self.raiz = None

    def insertar(self, est):
        if self.raiz is None:
            self.raiz = NodoABB(est)
            return
        actual = self.raiz
        while True:
            if est["id"] < actual.est["id"]:
                if actual.izq is None:
                    actual.izq = NodoABB(est)
                    return
                actual = actual.izq
            else:
                if actual.der is None:
                    actual.der = NodoABB(est)
                    return
                actual = actual.der

    def buscar(self, id_buscar):
        actual = self.raiz
        while actual:
            if id_buscar == actual.est["id"]:
                return actual.est
            if id_buscar < actual.est["id"]:
                actual = actual.izq
            else:
                actual = actual.der
        return None


# ---------- PARTE 4: BUSQUEDA EN LISTAS ----------

def buscar_en_lista(lista, id_buscar):
    for estudiante in lista:
        if estudiante["id"] == id_buscar:
            return estudiante
    return None


# ---------- EJECUCION Y PRUEBAS ----------

n_datos = 10000
ordenados = generar_estudiantes(n_datos)
aleatorios = ordenados.copy()
random.shuffle(aleatorios)

# Crear todas las versiones de los arboles
bplus_ord = BPlusTree()
bplus_rand = BPlusTree()
abb_ord = ABB()
abb_rand = ABB()

# Insertar datos
for e in ordenados:
    bplus_ord.insertar(e)
    abb_ord.insertar(e)

for e in aleatorios:
    bplus_rand.insertar(e)
    abb_rand.insertar(e)

# Generar  IDs para buscar
ids_a_buscar = []
for i in range(10000):
    ids_a_buscar.append(random.randint(1000, 1000 + n_datos - 1))


# --- MEDICIONES ---

# 1. Listas
inicio = time.perf_counter()
for i in ids_a_buscar: buscar_en_lista(ordenados, i)
t_lista_ord = time.perf_counter() - inicio

inicio = time.perf_counter()
for i in ids_a_buscar: buscar_en_lista(aleatorios, i)
t_lista_rand = time.perf_counter() - inicio

# 2. ABB
inicio = time.perf_counter()
for i in ids_a_buscar: abb_ord.buscar(i)
t_abb_ord = time.perf_counter() - inicio

inicio = time.perf_counter()
for i in ids_a_buscar: abb_rand.buscar(i)
t_abb_rand = time.perf_counter() - inicio

# 3. B+
inicio = time.perf_counter()
for i in ids_a_buscar: bplus_ord.buscar(i)
t_bplus_ord = time.perf_counter() - inicio

inicio = time.perf_counter()
for i in ids_a_buscar: bplus_rand.buscar(i)
t_bplus_rand = time.perf_counter() - inicio


# ---------- RESULTADOS FINALES ----------

print("--- RESULTADOS DE BUSQUEDA (N=" + str(n_datos) + ") ---")
print("Lista (Ordenada):        ", t_lista_ord)
print("Lista (Aleatoria):       ", t_lista_rand)
print("-" * 40)
print("ABB (Datos Ordenados):   ", t_abb_ord)
print("ABB (Datos Aleatorios):  ", t_abb_rand)
print("-" * 40)
print("Arbol B+ (Ordenado):     ", t_bplus_ord)
print("Arbol B+ (Aleatorio):    ", t_bplus_rand)


--- RESULTADOS DE BUSQUEDA (N=10000) ---
Lista (Ordenada):         2.6556052569999338
Lista (Aleatoria):        6.14515173399991
----------------------------------------
ABB (Datos Ordenados):    5.422883205999938
ABB (Datos Aleatorios):   0.04453988800003117
----------------------------------------
Arbol B+ (Ordenado):      0.042527718999963326
Arbol B+ (Aleatorio):     0.04112404499994682
